In [ ]:
import sys
import pandas as pd
path = "/home/aiuser/work/workspace/BigAlpha/data"
if path not in sys.path:
    sys.path.append(path)

from jyc_2026.cpt_jyc_2026_stock_barkm.builder import (
    CptJyc2026StockBar1mBuilder,
    CptJyc2026StockBarKmBuilder,
)
from utils import generate_dates_series
from datetime import datetime

dates_series = generate_dates_series(2020, 2024, "monthly")
dates_series.sort()
total_periods = len(dates_series)

# 构建1分钟K线
CptJyc2026StockBar1mBuilder('2020-01-01', '2024-12-31').build()


# 构建N分钟K线
for index, (start_date, end_date) in enumerate(dates_series, start=1):
    for K in [5, 15, 30]:
        now = datetime.now()
        CptJyc2026StockBarKmBuilder(start_date, end_date, K=K).build()
        duration = datetime.now() - now
        print(f"{K}m [{index}/{total_periods}] {100*index/total_periods:.2f}% | {start_date} 至 {end_date} | 耗时: {duration}")

In [ ]:
# 数据测试

from bigquant import dai
import numpy as np

date_str = '2020-01-02'
start_date = f'{date_str} 09:31:00'
end_date = f'{date_str} 23:59:00'

sql = 'SELECT * FROM cpt_jyc_2026_stock_bar1m'
df1 = dai.query(sql, filters={'date': [start_date, end_date]}).df()
instruments = df1['instrument'].unique().tolist()

sql = 'SELECT * FROM cn_stock_bar1m_derived_c'
df2 = dai.query(sql, filters={'date': [start_date, end_date], 'instrument': instruments}).df()

keys = ['date', 'instrument']
columns = [
    col for col in df1.columns
    if col not in keys + ['instrument_id', 'adjust_factor']
]

merged = df1.merge(
    df2[keys + columns], on=keys, how='left', suffixes=('_df1', '_df2')
)

for col in columns:
    left = merged[f'{col}_df1']
    right = merged[f'{col}_df2']

    if pd.api.types.is_numeric_dtype(left):
        same = np.isclose(left, right, rtol=0.01, equal_nan=True)
    else:
        same = left.eq(right) | (left.isna() & right.isna())

    diff_ratio = 1 - same.mean()
    assert diff_ratio <= 0.1, f'{col} 差异比例: {diff_ratio:.2%}'

print('数据一致')